<!-- parity-note -->
## MATLAB Parity Note
- Source MATLAB helpfile: `PPSimExample.mlx`
- Fidelity status: `partial`
- Remaining justified differences: The notebook now executes the full Python point-process simulation and analysis workflow without placeholders, but it still uses the native `CIFModel` path rather than the original MATLAB/Simulink recursive CIF model.

In [ ]:
# nSTAT-python notebook example: PPSimExample
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SRC_PATH = (REPO_ROOT / "src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nstat import Analysis, CIFModel, ConfigColl, CovColl, Covariate, FitResSummary, Trial, TrialConfig
from nstat.notebook_figures import FigureTracker

np.random.seed(5)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic='PPSimExample', output_root=OUTPUT_ROOT, expected_count=3)


def _figure(label: str, *, figsize=(8.5, 4.5)):
    fig = __tracker.new_figure(label)
    fig.clear()
    fig.set_size_inches(*figsize)
    return fig


def _logistic_rate(time, stimulus, mu=-3.0):
    dt = float(np.median(np.diff(time)))
    eta = mu + stimulus
    p = np.exp(np.clip(eta, -20.0, 20.0))
    p = p / (1.0 + p)
    return p / max(dt, 1e-12)


Ts = 0.001
tMin = 0.0
tMax = 10.0
t = np.arange(tMin, tMax + Ts, Ts)
mu = -3.0
stimulus_signal = np.sin(2 * np.pi * 1.0 * t)
baseline = Covariate(t, np.ones_like(t), "Baseline", "time", "s", "", ["mu"])
stim = Covariate(t, stimulus_signal, "Stimulus", "time", "s", "Voltage", ["sin"])
rate_hz = _logistic_rate(t, stimulus_signal, mu=mu)
lambda_model = CIFModel(t, rate_hz, name="lambda")
sC = lambda_model.simulate(num_realizations=5, seed=5)
cc = CovColl([stim, baseline])
trial = Trial(sC, cc)
print({"duration_s": tMax, "num_realizations": sC.numSpikeTrains, "mean_rate_hz": round(float(np.mean(rate_hz)), 3)})


In [ ]:
# SECTION 1: General Point Process Simulation
plt.close("all")


In [ ]:
# SECTION 2: Point Process Sample Path Generation
# This Python port uses a native CIFModel-driven rate simulation instead of the original MATLAB/Simulink model.


In [ ]:
# SECTION 3: History Effect
selfHist = [0.0, 0.001, 0.002, 0.003]
print({"history_windows_s": selfHist})


In [ ]:
# SECTION 4: Stimulus Effect
print({"stimulus_frequency_hz": 1.0, "stimulus_amplitude": 1.0})


In [ ]:
# SECTION 5: Ensemble Effect
print({"ensemble_effect": 0.0})


In [ ]:
# SECTION 6: Generate sample paths
fig = _figure("figure; subplot(2,1,1); sC.plot; subplot(2,1,2); stim.plot", figsize=(10.0, 5.5))
axs = fig.subplots(2, 1, sharex=True)
sC.plot(handle=axs[0])
axs[0].set_xlim(0.0, tMax / 5.0)
stim.plot(handle=axs[1])
axs[1].set_xlim(0.0, tMax / 5.0)


In [ ]:
# SECTION 7: GLM Model Fitting Setup
cfg = [
    TrialConfig([["Baseline", "mu"]], sampleRate=1.0 / Ts, name="Baseline"),
    TrialConfig([["Baseline", "mu"], ["Stimulus", "sin"]], sampleRate=1.0 / Ts, name="Stim"),
    TrialConfig([["Baseline", "mu"], ["Stimulus", "sin"]], sampleRate=1.0 / Ts, history=selfHist, name="Stim+Hist"),
]
cfgColl = ConfigColl(cfg)


In [ ]:
# SECTION 8: GLM Model Fitting and Results
results = Analysis.RunAnalysisForAllNeurons(trial, cfgColl)
fig = _figure("results{1}.plotResults", figsize=(11.0, 8.0))
results[0].plotResults(handle=fig)


In [ ]:
# SECTION 9: Results for across all sample paths
summary = FitResSummary(results)
fig = _figure("Summary.plotSummary", figsize=(10.0, 4.5))
summary.plotSummary(handle=fig)
print({"fit_names": summary.fitNames, "mean_AIC": np.asarray(summary.AIC, dtype=float).round(3).tolist()})
__tracker.finalize()
